# Amaliyot 11. Masofalar, oʻxshashlik, sentroidlar

11-interaktiv laboratoriyaga («Masofalar va sentroid») hamroh. Kurs mavzusi: matnlarni klasterlashning matematik asoslari

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.3, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Hujjatlar oʻxshashlik matritsalari

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

sample = pd.concat([df[df.label == l].head(3) for l in sorted(df.label.unique())]).reset_index(drop=True)
X = TfidfVectorizer().fit_transform(sample.text)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, M, t, cm in [(ax[0], cosine_similarity(X), "Kosinus oʻxshashlik", "Blues"), (ax[1], euclidean_distances(X), "Yevklid masofa", "Blues_r")]:
    im = a.imshow(M, cmap=cm); a.set_title(t); plt.colorbar(im, ax=a)
    a.set_xticks(range(len(sample))); a.set_xticklabels(sample.label, rotation=90); a.set_yticks(range(len(sample))); a.set_yticklabels(sample.label); a.grid(False)
plt.tight_layout(); plt.show()

## 2. Hujjat uzunligi: kosinus oʻzgarmaydi, Yevklid oʻzgaradi

In [ ]:
cv = CountVectorizer()
cv.fit(["oʻyin jamoa gol oʻyinchi qonun deputat"])
a = cv.transform(["oʻyin jamoa gol"]); b = cv.transform(["oʻyin jamoa gol oʻyin jamoa gol"])
print("Kosinus oʻxshashlik:", cosine_similarity(a, b)[0, 0].round(3))
print("Yevklid masofa:", euclidean_distances(a, b)[0, 0].round(3))

## 3. Sinf sentroidlari va eng yaqin sentroid boʻyicha tasniflash

In [ ]:
vec = TfidfVectorizer(); Xall = vec.fit_transform(df.text)
classes = sorted(df.label.unique())
cent = np.vstack([np.asarray(Xall[(df.label == c).values].mean(axis=0)) for c in classes])
pred = np.array(classes)[cosine_similarity(Xall, cent).argmax(axis=1)]
print("Eng yaqin sentroid boʻyicha accuracy:", round((pred == df.label.values).mean(), 3))
terms = np.array(vec.get_feature_names_out())
for c, row in zip(classes, cent):
    print(f"{c:12s}", ", ".join(terms[row.argsort()[::-1][:6]]))

## Topshiriqlar

1. TF-IDF ni quyidagiga almashtiring: `CountVectorizer`. Oʻxshashlik matritsalari qanday oʻzgardi?
2. Korpusga «chetga chiqqan nuqtalar» (juda uzun matnlar) qoʻshing va sentroid oʻrnini koordinatalar boʻyicha mediana bilan solishtiring.
3. Sonlar bilan isbotlang: L2-normallashtirilgan vektorlar uchun `d² = 2·(1 − cos)`.